In [15]:
import torch
import torch.nn as nn
from test3 import TransformerModel, generate_dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokens = ['0','1','2','3','4','5','6','7','8','9','+', '=','<PAD>','<BOS>','<EOS>']
vocab_size = len(tokens)

s2i = {c:i for i,c in enumerate(tokens)} # 从token到ids
i2s = {i:c for c,i in s2i.items()} # 从ids到token
# print (s2i)
# print (i2s)
PAD = s2i['<PAD>']
BOS = s2i['<BOS>']
EOS = s2i['<EOS>']

def tokenize(s):
    tokens = []
    i = 0
    while i < len(s):
        if s[i] == "<":
            j = s.index(">", i)
            tokens.append(s[i:j+1])
            i = j + 1
        else:
            tokens.append(s[i])
            i += 1
    return tokens


def encode_tokens(tokens):
    return [s2i[t] for t in tokens]

def pad(seq,length):
    return seq + [PAD]*(length-len(seq))


model = TransformerModel(vocab_size=vocab_size).to(device)
model.load_state_dict(
    torch.load("ckpts/transformer_lr=0.0001_epoch_15000.pt", map_location=device)
)

def predict(x_str, model):
    model.eval()

    # encoder 输入
    x = encode_tokens(tokenize(x_str))
    x = torch.tensor([pad(x, 11)]).to(device)

    y = [BOS]
    for _ in range(7):
        y_tensor = torch.tensor([pad(y, 8)]).to(device)
        with torch.no_grad():
            logits = model(x, y_tensor)
        next_token = logits[0, len(y) - 1].argmax().item()
        if next_token == EOS:
            break
        y.append(next_token)
    # 转回字符串
    result = "".join(i2s[i] for i in y[1:])
    return result


In [39]:
import random
from sklearn.model_selection import train_test_split
def generate_test(
        n_samples=100,
        max_digit=5,
        test_ratio=1,
        seed=42,
        digit_choices=[1]):

    random.seed(seed)
    train_x = []
    train_y = []

    xs = []
    ys = []

    for _ in range(n_samples):

        d1 = random.choice(digit_choices)
        d2 = random.choice(digit_choices)

        a = random.randint(10**(d1-1), 10**d1 - 1)
        b = random.randint(10**(d2-1), 10**d2 - 1)

        # encoder 输入 padding
        a_str = str(a).zfill(max_digit)
        b_str = str(b).zfill(max_digit)

        encoder_input = f"{a_str}+{b_str}"

        result = str(a+b)

        decoder_input = "<BOS>" + result
        decoder_target = result + "<EOS>"

        xs.append(encoder_input)
        ys.append((decoder_input, decoder_target))

    train_x, test_x, train_y, test_y = train_test_split(
        xs, ys, test_size=test_ratio, random_state=seed
    )

    return train_x, train_y, test_x, test_y

def calc_accuracy(xs, ys, model):
    correct = 0
    total = len(xs)

    for x, y in zip(xs, ys):

        pred = predict(x, model)

        # y = ('<BOS>104916', '104916<EOS>')
        true = y[1].replace("<EOS>", "")

        if pred == true:
            correct += 1

    return correct / total

def evaluate_digits(model):

    results = {}

    for d in range(1, 7):

        xs, ys, _, _ = generate_test(digit_choices=[d])

        acc = calc_accuracy(xs, ys, model)

        results[d] = acc

        print(f"{d}-digit accuracy: {acc:.4f}")

    return results

In [40]:
evaluate_digits(model)

1-digit accuracy: 0.0000
2-digit accuracy: 0.0404
3-digit accuracy: 1.0000
4-digit accuracy: 0.9899
5-digit accuracy: 0.8990
6-digit accuracy: 0.0000


{1: 0.0,
 2: 0.04040404040404041,
 3: 1.0,
 4: 0.98989898989899,
 5: 0.898989898989899,
 6: 0.0}